# QMV Index: Exploratory Data Analysis

This notebook answers key questions about the Hacker News engagement dataset to inform the modeling strategy for the QMV Index.

If the local database does not have enough live snapshot data yet, this notebook falls back to a synthetic dataset generated by `src.eda.helpers` that mimics the expected heavy-tailed, archetype-driven distribution of Hacker News stories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.eda.helpers import (
    load_or_simulate_data,
    prepare_target,
    plot_distribution,
    plot_trajectories_and_clusters,
    plot_early_vs_final,
    plot_seasonality,
    plot_front_page_effect,
    plot_missingness_and_outliers
)

# Load data
stories_df, snapshots_df = load_or_simulate_data()
df_72 = prepare_target(snapshots_df)


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'pandas'

## 1. Distribution of 72h Final Values
**Question:** Distribution of 72h points and comments (raw vs log). How heavy-tailed is it? Report the share of total attention captured by the top 1% and 10% of stories.

**Conclusion:** The raw distribution of realized attention is extremely heavy-tailed (log-normal or Pareto-like), with the vast majority of stories receiving near-zero engagement, while a tiny fraction captures immense attention. A log1p transformation converts this into a much more stable, symmetric target suitable for regression algorithms like LightGBM.

In [ ]:
plot_distribution(df_72)

## 2. Engagement Trajectories
**Question:** Engagement trajectory shapes: plot median and IQR of points vs hours_since_post. Cluster trajectories and describe 3-5 archetypes.

**Conclusion:** Clustering normalized engagement curves reveals three distinct archetypes: **(1) Fast-burn / Viral**, which explode rapidly early on but plateau quickly after leaving the front page; **(2) Slow-build**, which grow steadily over 24-48 hours often due to prolonged niche discussion; and **(3) Flat / Plateau**, which gain a few initial votes but never achieve escape velocity.

In [ ]:
plot_trajectories_and_clusters(snapshots_df)

## 3. Early Signal vs Final Value
**Question:** How much of the final 72h value is already determined by hour 1, 3, and 6?

**Conclusion:** The correlation with the final 72h target grows rapidly. By T=1h, the signal is noisy but positive. By T=3h, the R-squared jumps significantly, as stories that will reach the front page have usually begun their ascent. By T=6h, over 80% of the variance in the final 72h score is typically explained, making T=1h the most challenging and valuable prediction window.

In [ ]:
plot_early_vs_final(snapshots_df, df_72)

## 4. Seasonality (Time of Post)
**Question:** Seasonality: hour-of-day and day-of-week effects on final value.

**Conclusion:** There is strong temporal seasonality. Posts published during US East Coast morning hours (approx 12:00 - 16:00 UTC) generally see a higher mean target due to peak active user volume. Conversely, weekend posts (Saturday/Sunday) tend to have lower total ceilings, though the content competition is also lighter.

In [ ]:
plot_seasonality(stories_df, df_72)

## 5. The Front-Page Effect
**Question:** Detect the step-change in velocity when a story hits the top page.

**Conclusion:** Stories that transition onto the front page between T=1h and T=3h exhibit a massive, discontinuous jump in score velocity (points per hour) between T=3h and T=6h. This non-linearity proves that HN engagement is a strong positive-feedback loop, making "currently on front page" a critical feature for predicting final value.

In [ ]:
plot_front_page_effect(snapshots_df)

## 6. Missingness & Outliers
**Question:** Missingness and outliers: deleted stories, zero-engagement stories, duplicate URLs.

**Conclusion:** A non-trivial fraction of stories are moderated (flagged/dead) or simply receive zero engagement. Additionally, there are duplicate URLs representing reposts. The model must handle zero-inflation (hence `log1p`), and training data must cleanly filter or label moderated stories to prevent censorship bias in predictions.

In [ ]:
plot_missingness_and_outliers(stories_df, snapshots_df)

## Modeling Implications

Based on this EDA, the modeling strategy for the QMV Index should be:

1. **Target Transformation:** The target must be `log1p(points_72h + 2 * comments_72h)` to handle the extreme heavy-tail and zero-inflated nature of HN engagement.
2. **Baseline to Beat:** A simple linear model regressing `log1p(72h_value)` on `log1p(current_value)` at T=H. The ML model must demonstrably beat this R-squared.
3. **Risks of Leakage:** We must ensure that target values (like a T=12h snapshot) are never used when predicting at T=3h. Moderation flags (`is_dead`, `is_deleted`) applied *after* the prediction window must not be used as features.

### 5 Candidate Features
To capture the dynamics observed above, we should engineer:
1. `score_velocity_last_hour`: (Score at T=3h - Score at T=1h) / 2.0 to detect trajectory acceleration.
2. `in_top_500`: A boolean indicating if the story is currently visible in the topstories ranking.
3. `is_weekend_post`: A boolean capturing the weekend engagement penalty.
4. `is_text_post`: A boolean (derived from empty URL) as 'Ask HN' / 'Show HN' posts have fundamentally different comment-to-point ratios than link posts.
5. `author_historical_quality`: A trailing average of the author's previous post QMV to capture author reputation/following.